# 12 — Board Pack Reviewer
> Reads a board pack as a sceptical non-executive director would — ranking risks, naming what the pack omits, isolating decisions, and drafting the questions management would rather not hear.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


# --- Schema ---

class TopRisk(BaseModel):
    rank: int = Field(description="Priority rank: 1 is the highest concern")
    area: Literal[
        "financial", "operational", "strategic", "regulatory", "governance", "reputational"
    ]
    severity: Literal["critical", "high", "medium"]
    title: str
    detail: str = Field(description="What specifically is the risk and why it matters to the board")
    source_section: str = Field(description="Which section of the board pack this comes from")
    suggested_question: str = Field(
        description="Question a non-executive director should ask management on this risk"
    )


class InformationGap(BaseModel):
    section: str = Field(description="Which section has the gap")
    missing: str = Field(description="What information is absent or unclear")
    why_it_matters: str = Field(description="Why the board needs this before deciding")


class DecisionRequired(BaseModel):
    item: str = Field(description="The specific decision or approval the board is being asked to make")
    context: str = Field(description="Background a director needs to vote on this item")
    recommendation: Optional[str] = Field(
        default=None,
        description="Management's stated recommendation, if provided"
    )
    key_consideration: str = Field(
        description="The one thing a director must weigh before approving"
    )


class DirectorBriefing(BaseModel):
    """Structured board pack review framed for a non-executive director."""

    company: Optional[str] = None
    meeting_date: Optional[str] = None
    overall_pack_quality: Literal["strong", "adequate", "weak"] = Field(
        description="Overall quality of the board pack as a governance document"
    )
    executive_assessment: str = Field(
        description="3-4 sentence summary for a NED arriving five minutes before the meeting"
    )
    top_risks: List[TopRisk] = Field(
        description="Up to five risks ranked by severity -- framed as NED concerns, not management euphemisms"
    )
    information_gaps: List[InformationGap] = Field(
        description="Material information absent from the pack that the board needs"
    )
    decisions_required: List[DecisionRequired] = Field(
        description="Items requiring board approval or formal decision"
    )
    questions_for_management: List[str] = Field(
        description="Suggested questions for the meeting -- probing, not procedural"
    )


# --- Agent ---

REVIEWER_SYSTEM = SystemMessage(
    """You are an experienced non-executive director reviewing a board pack before a meeting.

Your job is to produce a structured briefing that helps fellow directors cut through
management language and focus on what actually matters.

Rules:
- Frame every risk as a board concern, not a management update
- Name information gaps explicitly -- "the pack does not disclose X" is more useful than silence
- Questions for management must be probing: challenge assumptions, not process
- overall_pack_quality reflects governance fitness, not length or formatting
- If something looks like it has been sanitised or is missing context, say so

You serve shareholders and stakeholders, not management."""
)


def run(board_pack_text: str) -> DirectorBriefing:
    """
    Single-agent board pack reviewer: reads the full pack, returns a structured
    DirectorBriefing framed for a non-executive director.
    """
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    reviewer = REVIEWER_SYSTEM | llm.with_structured_output(DirectorBriefing)
    return reviewer.invoke(
        HumanMessage(content="Board pack to review:\n\n" + board_pack_text)
    )


print("Ready.")

## The scenario

We're reviewing the board pack for Meridian Capital Partners ahead of their Q2 investment committee meeting. The pack covers a request to deploy £40m into a late-stage proptech startup, a review of two existing portfolio companies showing performance slippage, and a proposed change to the fund's carried interest structure. The agent will rank the risks, surface what the pack omits, and prepare the questions the IC should put to the deal team before any capital is committed.

In [ ]:
BOARD_PACK = """
MERIDIAN CAPITAL PARTNERS LLP
INVESTMENT COMMITTEE MEETING
17 June 2025 | 14:00 BST

AGENDA
1. Minutes of previous meeting (approval)
2. Managing Partner Update
3. New Investment Proposal -- Stackr Technologies Ltd (Series C)
4. Portfolio Review -- Flagged Holdings
5. Proposed amendment to carried interest allocation
6. Any other business

---

ITEM 2: MANAGING PARTNER UPDATE

Fund III is now 78% deployed with 14 months remaining in the investment period.
LP relations remain stable. One LP (representing 12% of committed capital) has
requested a side letter amendment regarding co-investment rights. Negotiations
are ongoing; no further detail is provided in this pack.

The committee is asked to note the update.

---

ITEM 3: NEW INVESTMENT PROPOSAL -- STACKR TECHNOLOGIES LTD

The deal team proposes a GBP 40m Series C investment in Stackr Technologies Ltd,
a B2B proptech platform providing AI-driven property valuation and lease management
software to commercial real estate operators.

Stackr FY2024 financials (unaudited management accounts):
  ARR:                     GBP 18.2m  (+61% YoY)
  Gross margin:            67%
  Net revenue retention:   108%
  Cash burn:               GBP 1.9m/month
  Runway:                  8 months at current burn

Proposed valuation: GBP 210m pre-money (11.5x ARR)
Meridian ownership post-close: 16.4%

The deal team notes that two other funds are in process and a term sheet
deadline has been set for 30 June 2025. No independent financial due diligence
has been completed. No reference calls with customers are included in the pack.
The lead partner on this deal joined Meridian four months ago.

The committee is asked to approve the investment in principle, with final
legal and financial due diligence to be completed post-approval.

---

ITEM 4: PORTFOLIO REVIEW -- FLAGGED HOLDINGS

Two portfolio companies have been flagged for committee attention.

Orbivault Ltd (Fund II, invested 2021)
  Revenue vs plan: -34% in Q1 2025
  Current cash position: GBP 0.9m (approximately 3 months runway)
  CEO resigned in March 2025. Interim CEO appointed internally.
  The company is exploring bridge financing options. No term sheet in place.
  No down-round valuation has been reflected in Fund II NAV. Reason not stated.

Cerela Health (Fund III, invested 2023)
  Regulatory approval for primary product delayed by 9 months (FDA request
  for additional clinical data received February 2025).
  Cash runway: 11 months.
  Management believes approval will be received by Q1 2026. No independent
  regulatory assessment is referenced.

The committee is asked to note the portfolio update.

---

ITEM 5: CARRIED INTEREST ALLOCATION AMENDMENT

Management proposes to amend the carried interest allocation for Fund III to
increase the share allocated to the investment team from 70% to 80%, reducing
the management company's share from 30% to 20%.

The rationale given is retention of senior deal team members following two
departures in H2 2024. LP consent requirements under the LPA are not addressed
in this pack. The GP's legal counsel has not provided a written opinion.

The committee is asked to approve the amendment.
"""

briefing = run(BOARD_PACK)

SEV = {"critical": "CRIT", "high": "HIGH", "medium": "MED "}

print("=" * 65)
print(f"DIRECTOR BRIEFING  |  Pack quality: {briefing.overall_pack_quality.upper()}")
if briefing.company:
    print(f"Company: {briefing.company}")
if briefing.meeting_date:
    print(f"Meeting: {briefing.meeting_date}")
print("=" * 65)

print(f"\n{briefing.executive_assessment}")

print(f"\nTOP RISKS ({len(briefing.top_risks)}):")
for r in briefing.top_risks:
    print(f"\n  [{r.rank}] [{SEV[r.severity]}] [{r.area.upper()}] {r.title}")
    print(f"      Source: {r.source_section}")
    print(f"      {r.detail}")
    print(f"      Q: {r.suggested_question}")

if briefing.information_gaps:
    print(f"\nINFORMATION GAPS ({len(briefing.information_gaps)}):")
    for g in briefing.information_gaps:
        print(f"\n  [{g.section}] {g.missing}")
        print(f"  Why it matters: {g.why_it_matters}")

if briefing.decisions_required:
    print(f"\nDECISIONS REQUIRED ({len(briefing.decisions_required)}):")
    for d in briefing.decisions_required:
        print(f"\n  DECISION: {d.item}")
        print(f"  Context: {d.context}")
        if d.recommendation:
            print(f"  Management recommends: {d.recommendation}")
        print(f"  Key consideration: {d.key_consideration}")

if briefing.questions_for_management:
    print("\nSUGGESTED QUESTIONS FOR MANAGEMENT:")
    for i, q in enumerate(briefing.questions_for_management, 1):
        print(f"  {i}. {q}")

## Use your own data

Replace the input above with:
- Your board pack text (CEO update, financials, any agenda items requiring approval)
- Any section headings or summaries — it does not need to be a formal document

The agent will return ranked risks with source citations, explicit information gaps, a list of decisions requiring your vote, and questions calibrated to challenge management's assumptions.